# P1b: freeze the calibration-only router state

This CPU-only prerequisite verifies P0b and the full private P1a report, selects one complete-coverage constant policy per task–backbone group, freezes per-series retained-policy utility scores, and hashes the resulting state. It performs **no model inference** and does not instantiate sealed origins.

Use a **CPU runtime**. Send the compact final JSON for review before running either GPU notebook.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests'
)
P0B_ROOT = PRIVATE_ROOT / 'p0b'
P1A_ROOT = PRIVATE_ROOT / 'p1a'
P1B_ROOT = PRIVATE_ROOT / 'p1b'
assert P0B_ROOT.exists(), 'P0b artifacts are missing from Google Drive.'
assert (P1A_ROOT / 'reports/p1a_applicability_audit.json').exists(), (
    'The full P1a report is missing from Google Drive.'
)
P1B_ROOT.mkdir(parents=True, exist_ok=True)
print('P0b input root:', P0B_ROOT)
print('P1a input root:', P1A_ROOT)
print('P1b durable root:', P1B_ROOT)

In [ ]:
import json

from covsafe.p1b import EXPECTED_P1B_CONFIG_HASH, run_p1b_router_state

print('Frozen P1b config hash:', EXPECTED_P1B_CONFIG_HASH)
report = run_p1b_router_state(REPO, P0B_ROOT, P1A_ROOT, P1B_ROOT)
print('Router state created and stored durably.')

In [ ]:
compact = {
    'config_hash': report['config_hash'],
    'git_commit': report['git_commit'],
    'scientific_code_sha256': report['scientific_code_sha256'],
    'scope': report['scope'],
    'applicability_map': report['applicability_map'],
    'constant_policy_by_group': {
        backbone: {
            task: {
                'fullset': values['fullset_best_policy_id'],
                'binary': values['binary_best_policy_id'],
                'calibration_units': values['calibration_unit_count'],
            }
            for task, values in tasks.items()
        }
        for backbone, tasks in report['constant_policies'].items()
    },
    'router_score_artifact': report['router_score_artifact'],
    'source_artifacts': report['source_artifacts'],
    'authorization': report['authorization'],
}
print(json.dumps(compact, indent=2, ensure_ascii=False))

## Stop here

Send the compact JSON above. Do not run notebooks 10–12 until this state has been reviewed.